## Content-Based Recommender | Model 4 | Sentence Transformers with Weighted Embeddings Blended

### Recommends based off the movies's plot, creditcs, genre and keywords via Sentence Transformers

###### Inspired by: Simulating Ibtesam
###### Link: https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system/notebook

In [27]:
# Imports
import pandas as pd
import numpy as np
import random
from collections import Counter
from ast import literal_eval
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [28]:
#%pip install sentence-transformers

In [29]:
# Read csv
df1=pd.read_csv('../tmdb/tmdb_5000_credits.csv')
df2=pd.read_csv('../tmdb/tmdb_5000_movies.csv')

In [30]:
# Join two datasets on id column
df1.columns = ['id','tittle','cast','crew']
df2= df2.merge(df1,on='id')

In [31]:
# Parse the stringified features into their corresponding python objects
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(literal_eval)

KeyboardInterrupt: 

#### Functions that will help extract required info from each feature

In [ ]:
# Get the director's name from the crew feature. If director is not listed, return NaN
def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [ ]:
# Test directors function
df2['director'] = df2['crew'].apply(get_director)
df2[['title', 'director']].head()

,title,director
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


In [ ]:
# Returns the list top 3 elements or entire list; whichever is more.
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        
        # Check if more than 3 elements exist. If yes, return only first three. If no, return entire list.
        if len(names) > 3:
            names = names[:3]
        return names

    # Return empty list in case of missing/malformed data
    return []

In [ ]:
# Define new director, cast, genres and keywords features that are in a suitable form.
df2['director'] = df2['crew'].apply(get_director)

features = ['cast', 'keywords', 'genres']
for feature in features:
    df2[feature] = df2[feature].apply(get_list)

In [ ]:
# Print the new features of the first 3 films
df2[['title', 'cast', 'director', 'keywords', 'genres']].head(3)

,title,cast,director,keywords,genres
0,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,"[culture clash, future, space war]","[Action, Adventure, Fantasy]"
1,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,"[ocean, drug abuse, exotic island]","[Adventure, Fantasy, Action]"
2,Spectre,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,"[spy, based on novel, secret agent]","[Action, Adventure, Crime]"


#### Convert names and keywords instances into lowercase and strip spaces between them so vectorizer doesn't get confused by multiple people with the same first or last name.

In [ ]:
# Function to convert all strings to lower case and strip names of spaces
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        # Check if director exists. If not, return empty string
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        else:
            return ''

In [ ]:
# Apply clean_data function to your features.
features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df2[feature] = df2[feature].apply(clean_data)

## Recommendation function

In [ ]:
# # Function that takes a movie title and returns the 15 most similar movies based on neural similarity matrix.
# def get_recommendations(title):
    
#     # Make title all lowsercase
#     title = title.lower()

#     # Get copy of indices and make them lowercase to compare to title
#     indices_lower = indices.copy()
#     indices_lower.index = indices_lower.index.str.lower()

#     # Return message if movie is not in list
#     if title not in indices_lower:
#         return f"Movie '{title}' not found in database."

#     # get index of title in indices
#     idx = indices_lower[title]

#     # Get similarity scores for this movie against all others
#     sim_scores = list(enumerate(cosine_sim_nn[idx]))
    
#     # Sort by score descending
#     sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
#     # Skip index 0 (the movie itself), and print top 10
#     sim_scores = sim_scores[1:11]

#     movie_indices = [i[0] for i in sim_scores]
#     return df2['title'].iloc[movie_indices]

## Prepare for Evaluation

#### Test users and movies they enjoy

In [ ]:
# dictionary of 5 users with difference preferences
# Each movie is considered a movie the user enjoyed

user_history = {
    # User 1: Classic gangster & crime dramas
    "user_1": [
        "The Godfather", "GoodFellas", "Scarface", "Pulp Fiction", "The Departed",
        "The Godfather: Part II", "Casino", "Donnie Brasco", "Once Upon a Time in America",
        "The Untouchables", "Road to Perdition", "Public Enemies", "Gangs of New York",
        "A History of Violence", "Eastern Promises", "The Town", "The Conformist",
        "Find Me Guilty", "Black Mass", "The Hills Have Eyes"
    ],

    # User 2: Blockbuster action & superhero movies
    "user_2": [
        "Avatar", "Titanic", "Avengers: Age of Ultron", "Guardians of the Galaxy", "Iron Man",
        "Thor", "Captain America: The First Avenger", "The Avengers", "Ant-Man",
        "The Incredible Hulk", "Captain America: Civil War", "Iron Man 2", "Iron Man 3",
        "Thor: The Dark World", "Batman Begins", "The Dark Knight", "The Dark Knight Rises",
        "Batman & Robin", "Batman Returns", "Batman v Superman: Dawn of Justice"
    ],

    # User 3: Musical & biographical movies
    "user_3": [
        "Chicago", "Moulin Rouge!", "8MM", "Amnesiac", "Grease",
        "Les Misérables", "Inception", "The Pursuit of Happyness", "The Hit List",
        "Singin' in the Rain", "The Sound of Music", "West Side Story", "Mary Poppins",
        "The Wizard of Oz", "Frozen", "Aladdin", "Cinderella", "The Nutcracker",
        "Alice in Wonderland", "The Broadway Melody"
    ],

    # User 4: Fantasy & young adult series
    "user_4": [
        "The Lord of the Rings: The Fellowship of the Ring", "The Hobbit: An Unexpected Journey",
        "The Lord of the Rings: The Two Towers", "The Lord of the Rings: The Return of the King",
        "Harry Potter and the Philosopher's Stone", "Harry Potter and the Chamber of Secrets",
        "Harry Potter and the Prisoner of Azkaban", "The Hunger Games: Catching Fire",
        "The Hunger Games: Mockingjay - Part 2", "The Twilight Saga: New Moon",
        "The Twilight Saga: Eclipse", "The Twilight Saga: Breaking Dawn - Part 2",
        "Percy Jackson: Sea of Monsters", "Percy Jackson & the Olympians: The Lightning Thief",
        "Harry Potter and the Order of the Phoenix", "The Chronicles of Narnia: The Lion, the Witch and the Wardrobe",
        "The Hobbit: The Desolation of Smaug", "The Hobbit: The Battle of the Five Armies",
        "The Adventures of Huck Finn", "Hellboy II: The Golden Army"
    ],

    # User 5: Horror & thriller movies
    "user_5": [
        "The Shining", "1408", "8 Days", "The Conjuring", "Insidious",
        "Sinister", "Annabelle", "Paranormal Activity 2", "Halloween: Resurrection", "Psycho",
        "Jaws", "Saw: The Final Chapter", "Scream 3", "Pet Sematary", "White Noise 2: The Light",
        "It Follows", "The Possession", "The Exorcist", "Evil Dead", "Restoration"
    ]
}

## Split data into known and unknown

In [ ]:
# Split movies into training and testing to evaluate model (5 training, 15 testing)
train_test_split = {}
split_ratio = 5

# Same results
random.seed(7)

# list and dict to store each unknown movie for each user
#users = {}

for user, movies in user_history.items():
    
    # List of unknown movies
    unknown_movies = []
    
    # add movies to either known or unknown list
    known = random.sample(movies, split_ratio)
    unknown = [m for m in movies if m not in known]
    
    # Add known and unknow movies to dictionary for corresponding user
    train_test_split[user] = {"known": known, "unknown": unknown}
    
    # append unknown movies to list
    #unknown_movies.append(unknown)
    #users[user] = unknown_movies

# Print 
for user, split in train_test_split.items():
    print(f"{user}:")
    print("Known:", split["known"])
    print("Unknown:", split["unknown"])
    print()

user_1:
Known: ['Road to Perdition', 'The Departed', 'Gangs of New York', 'GoodFellas', 'Scarface']
Unknown: ['The Godfather', 'Pulp Fiction', 'The Godfather: Part II', 'Casino', 'Donnie Brasco', 'Once Upon a Time in America', 'The Untouchables', 'Public Enemies', 'A History of Violence', 'Eastern Promises', 'The Town', 'The Conformist', 'Find Me Guilty', 'Black Mass', 'The Hills Have Eyes']

user_2:
Known: ['Batman & Robin', 'Guardians of the Galaxy', 'Iron Man 2', 'Titanic', 'Captain America: The First Avenger']
Unknown: ['Avatar', 'Avengers: Age of Ultron', 'Iron Man', 'Thor', 'The Avengers', 'Ant-Man', 'The Incredible Hulk', 'Captain America: Civil War', 'Iron Man 3', 'Thor: The Dark World', 'Batman Begins', 'The Dark Knight', 'The Dark Knight Rises', 'Batman Returns', 'Batman v Superman: Dawn of Justice']

user_3:
Known: ['Moulin Rouge!', '8MM', 'The Wizard of Oz', 'The Nutcracker', 'Alice in Wonderland']
Unknown: ['Chicago', 'Amnesiac', 'Grease', 'Les Misérables', 'Inception', 'T

## Evaluation

In [ ]:
# Load pre-trained neural model
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def evaluate_exp(train_test_split, get_recs_fn, k=10):
    results = {}
    for user, data in train_test_split.items():
        known_movies   = data["known"]
        unknown_movies = data["unknown"]

        all_recommendations = []
        for movie in known_movies:
            recs = get_recs_fn(movie)
            if isinstance(recs, str):
                continue
            all_recommendations.extend(recs.tolist())

        movie_counts = Counter(all_recommendations)
        top_k = [movie for movie, _ in movie_counts.most_common(k)]

        hits      = sum(1 for movie in top_k if movie in unknown_movies)
        precision = hits / k
        recall    = hits / len(unknown_movies)
        results[user] = {"precision": precision, "recall": recall}

    avg_p = sum(v["precision"] for v in results.values()) / len(results)
    avg_r = sum(v["recall"]    for v in results.values()) / len(results)
    print(f"  Avg Precision@{k}: {avg_p:.2f} | Avg Recall@{k}: {avg_r:.2f}")
    return results

In [ ]:
# need to fix this, should be function calles nothing definitions within functions.

def run_blend_experiment(df2, train_test_split, plot_weight=0.7, meta_weight=0.3, k=10):
    print(f"\n=== Embedding Blend (plot={plot_weight}, meta={meta_weight}) ===")
    df_exp = df2.copy()

    # Separate text fields
    def get_plot(x):
        return x['overview'] if isinstance(x['overview'], str) else ''

    def get_meta(x):
        cast     = ' '.join(x['cast'])     if isinstance(x['cast'],      list) else ''
        genres   = ' '.join(x['genres'])   if isinstance(x['genres'],    list) else ''
        keywords = ' '.join(x['keywords']) if isinstance(x['keywords'],  list) else ''
        director = x['director']           if isinstance(x['director'],  str)  else ''
        return f"{cast} {director} {genres} {keywords}"

    plots = df_exp.apply(get_plot, axis=1).tolist()
    metas = df_exp.apply(get_meta, axis=1).tolist()

    # Encode separately, then blend
    plot_embeddings = model.encode(plots, show_progress_bar=False, batch_size=64)
    meta_embeddings = model.encode(metas, show_progress_bar=False, batch_size=64)

    blended = plot_weight * plot_embeddings + meta_weight * meta_embeddings

    cosine_sim_blend = cosine_similarity(blended, blended)
    indices_blend = pd.Series(df_exp.index, index=df_exp['title']).drop_duplicates()

    def get_recommendations_blend(title):
        title = title.lower()
        indices_lower = indices_blend.copy()
        indices_lower.index = indices_lower.index.str.lower()
        if title not in indices_lower:
            return f"Movie '{title}' not found."
        idx = indices_lower[title]
        sim_scores = sorted(enumerate(cosine_sim_blend[idx]), key=lambda x: x[1], reverse=True)[1:11]
        return df_exp['title'].iloc[[i[0] for i in sim_scores]]

    return evaluate_exp(train_test_split, get_recommendations_blend, k=k)

In [ ]:
# Run across different blend ratios
for pw, mw in [(1.0, 0.0), (0.7, 0.3), (0.5, 0.5), (0.3, 0.7)]:
    run_blend_experiment(df2, train_test_split, plot_weight=pw, meta_weight=mw)